# Project 5 - Topic Screening
This notebook provides a starting point for the **Topic screening** project.

We use the **Synergy** datasets to train a baseline classifier that identifies potentially toxic comments using TF-IDF and Logistic Regression.

You are expected to extend this baseline through your own experiments, following an **evidence-driven model development approach**. Refer to the project brief for the full project objectives and requirements.

## 1. Your Tasks

The provided notebook establishes a simple baseline for detecting potentially toxic comments. Your task is to **evaluate this baseline, identify its limitations, and investigate possible improvements**.

### 1.1 Explore the Dataset

Get familiar with the data and the classification task.

- Examine the dataset characteristics, class distribution, and representative examples.
- Consider how the target labels are defined and whether the data is suitable for the intended application.
- Identify potential issues related to data quality, coverage, or representativeness.

### 1.2 Evaluate the Baseline

Determine how well the model performs and which evaluation criteria are appropriate.

- Select and justify metrics relevant to the task.
- Evaluate overall and class-wise performance.
- Consider the consequences of different types of errors, particularly false positives and false negatives.

### 1.3 Analyze Errors and Diagnose Limitations

Investigate where and why the classifier makes mistakes.

- Examine misclassified examples and identify recurring patterns.
- Formulate possible explanations for the observed errors.
- Distinguish observations supported by the data from hypotheses that require further investigation.

### 1.4 Investigate Possible Improvements

Based on your observations and relevant literature, identify promising directions for improving the baseline.

- Research alternative approaches, representations, or model configurations.
- Explain which limitations these approaches might address.
- Where feasible, implement and evaluate an improvement against the original baseline using a consistent evaluation setup.

### 1.5 Document Your Findings

Summarize your experiments and the reasoning behind your decisions.

- What did you investigate, and why?
- What evidence supports your observations and conclusions?
- Which improvements appear promising, and what remains unresolved?

**Important:** Your decisions should be grounded in **empirical evidence or relevant research**, rather than assumptions. When proposing changes, explain the motivation and evaluate their effects whenever possible.

Use the validation set for model development and comparisons. Keep the test set reserved for final evaluation.

## 2. Starter code

We use the **Nagtegaal_2019** dataset, which papers selected during the screening of title/abstracts of a literature review, as well as those papers that were actually relevant after reading the full text. As part of the screening, we are only interested on what we can derive from the title/abstract. This info is in:

`label_abstract_screening`	

### 2.1 Loading the dataset and preparing splits

If your Jupyter notebook cannot access the Internet (e.g., on the YourAI cluster), download the dataset beforehand using the `download_dataset.py` script:

````bash
python scripts/download_sciebo.py "https://hsbi.sciebo.de/s/agEfPKycnsM6N69" "data/synergy/"
````

In [1]:
import pandas as pd

df = pd.read_csv("../../data/synergy/Nagtegaal_2019.csv")

print("Dataset size:", df.shape)
display(df.head())

Dataset size: (600, 6)


,record_id,title,abstract,label_included,label_abstract_screening,duplicate_record_id
0,1,"A prospective, controlled trial of a pharmacy-...",Although venous thromboembolism is an importan...,0,1,NaN
1,3,Using peer feedback to improve handwashing and...,Background: Without protective practices such ...,0,1,NaN
2,8,"""If it ain't broke, don't fix it",NaN,0,0,NaN
3,13,100 Apples Divided by 15 Red Herrings: A Cauti...,"In 1863, Florence Nightingale argued that Lond...",0,0,NaN
4,16,2015 AAHA/AAFP pain management guidelines for ...,RATIONALE: The robust advances in pain managem...,0,0,NaN


**Preparing splits**. The dataset doesn't have them. 

In [7]:
from sklearn.model_selection import train_test_split

SEED = 42
# We set label to the actual screening-label column
df["label"] = df["label_abstract_screening"]
df["text"] = (
    df["title"].fillna("") + ". " +
    df["abstract"].fillna("")
)

# 60% training, 40% held out
train_df, temp_df = train_test_split(
    df, test_size=0.4, stratify=df["label"], random_state=SEED
)

# Divide held-out data equally into validation and test
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Training: 360
Validation: 120
Test: 120


### 2.2 Establishing a Baseline Classifier

Our baseline uses:

- **TF-IDF:** Represents comments using weighted word and bigram features.
- **Logistic Regression:** Predicts whether a comment belongs to the toxic or non-toxic class.

We use `class_weight="balanced"` to account for class imbalance during training.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

# Text representation
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
y_train = train_df["label"]
y_val = val_df["label"]

# Train classifier
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)

# Predictions on validation data
y_proba = model.predict_proba(X_val)[:, 1]
y_pred = model.predict(X_val)

print("Training completed.")
print("Number of features:", X_train.shape[1])


Training completed.
Number of features: 12589


### 2.3 Initial Evaluation

In [9]:
from sklearn.metrics import accuracy_score, f1_score

print (accuracy_score(y_val, y_pred))
print (f1_score(y_val, y_pred, average="macro"))

0.875
0.7875103293589896


## 3. Your experiments